In [0]:
%python
#Instalação das bibliotecas necessárias
!pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os, glob, shutil

In [0]:
%python

# Consumo das APIs do Kaggle e download dos dados. 
# Criamos um volume para armazenar os dados em csv para que possamos manter os arquivos disponíveis em seu formato original.
# Criamos as tabelas em delta no schema bronze.

catalog = "pospucrio"
schema  = "bronze"
volume  = "raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

local_path  = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
volume_path = f"/Volumes/{catalog}/{schema}/{volume}"

for csv in glob.glob(os.path.join(local_path, "*.csv")):
    arquivo = os.path.basename(csv)
    tabela  = arquivo.replace(".csv", "")

    shutil.copy(csv, f"{volume_path}/{arquivo}")

    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(f"{volume_path}/{arquivo}"))

    df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{tabela}")
    print(f"{tabela}: {df.count()} linhas")